Honestly, I was at a loss when I made a book project, printed, folded and cut it, just to realize it had only even pages, repeating front and back. That was quite frustrating, and I tried it over and over to make it work on both bookbinder.app and bookbinder.js, but they would always output the wrong thing.

So I thought to myself that the next best thing would be to implement something myself that would do both things all at once:
- Organize pages;
- Impose it in a PDF;

Thankfully, I knew that pymupdf's documentation for 4-up'ing a pdf was a pretty good hint, and I just needed to understand how to organize the pages.

I'll admit that I was weak at this part, however, and say it outright that I didn't code any of this. I asked DeepSeek for help and within an hour I had a working prototype, giving it the pymupdf code as a hint for the final step. The only different thing here is that it works best with a signature length of 24 (or 6 sheets per signature), and can't work at a custom amount anymore. So you need to input a number that's multiple of 8, or it will add a lot of padding to the signatures, increasing the number of pages a lot.

It might be fine, but I'm quite used to 5 sheets per signature. Honestly it's probably not that big of a deal. If it were 8 sheets, however, it would be too bulky for my own taste. I'll also try to make it work with 5 in another slop round, and if I manage to do that I'll add another notebook to this project just for that. 

In [ ]:
book_name = "book_name" # insert pdf/book name here. this is what will be read in the code, so please input it properly.

In [2]:
def quarto_imposition(n, sig_size=24):
    """
    Return list of (front, back) tuples, one per physical A4 sheet.
    Each of `front` / `back` is a 4-tuple (TL, TR, BL, BR) of 1-based page
    numbers, or 0 for a blank slot.

    Signatures are padded to a multiple of 8 pages (quarto needs 8 pages
    per physical sheet: 4 per side).
    """
    sheets = []
    start = 1
    while start <= n:
        remaining = n - start + 1
        size = min(sig_size, remaining)
        padded = ((size + 7) // 8) * 8        # pad to multiple of 8

        def pg(local, start=start, size=size):
            return start + local - 1 if 1 <= local <= size else 0

        for k in range(padded // 8):
            front = (pg(padded - 4*k),      # TL
                     pg(4*k + 1),           # TR
                     pg(padded - 4*k - 2),  # BL
                     pg(4*k + 3))           # BR
            back  = (pg(4*k + 2),           # TL
                     pg(padded - 4*k - 1),  # TR
                     pg(4*k + 4),           # BL
                     pg(padded - 4*k - 3))  # BR
            sheets.append((front, back))

        start += size
    return sheets

In [5]:
import pymupdf

src = pymupdf.open(f"{book_name}.pdf")
n = len(src)

front_doc = pymupdf.open()
back_doc  = pymupdf.open()

# --- same rectangle setup as your code ---
width, height = pymupdf.paper_size("a4")
r = pymupdf.Rect(0, 0, width, height)
r1 = r / 2
r2 = r1 + (r1.width, 0, r1.width, 0)
r3 = r1 + (0, r1.height, 0, r1.height)
r4 = pymupdf.Rect(r1.br, r.br)
r_tab = [r1, r2, r3, r4]        # TL, TR, BL, BR

def make_page(doc, quad):
    page = doc.new_page(-1, width=width, height=height)
    for rect, pno in zip(r_tab, quad):
        if pno == 0:
            continue                # blank slot
        page.show_pdf_page(rect, src, pno - 1)   # source is 0-indexed

for front, back in quarto_imposition(n, sig_size=24):
    make_page(front_doc, front)
    make_page(back_doc,  back)

front_doc.save("fronts.pdf", garbage=3, deflate=True)
back_doc .save("backs.pdf",  garbage=3, deflate=True)

In [6]:
combined = pymupdf.open()
for front, back in quarto_imposition(n, sig_size=24):
    make_page(combined, front)
    make_page(combined, back)
combined.save(f"[final] {book_name}.pdf", garbage=3, deflate=True)